In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
appName("week4-assignments 3"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
# Cases=====  /public/trendytech/covid19/cases/covid_dataset_cases.csv
# date,state,positive,negative,pending,hospitalizedCurrently,hospitalizedCumulative,inIcuCurrently,inIcuCumulative,onVentilatorCurrently,onVentilatorCumulative,recovered,
# dataQualityGrade,lastUpdateEt,dateModified,checkTimeEt,death,hospitalized,dateChecked,totalTestsViral,positiveTestsViral,negativeTestsViral,positiveCasesViral,deathConfirmed,deathProbable,
# fips,positiveIncrease,negativeIncrease,total,totalTestResults,totalTestResultsIncrease,posNeg,deathIncrease,hospitalizedIncrease,hash,commercialScore,negativeRegularScore,
# negativeScore,positiveScore,score,grade

In [3]:
# States===== /public/trendytech/covid19/states/covid_dataset_states.csv
# state,notes,covid19Site,covid19SiteSecondary,covid19SiteTertiary,twitter,covid19SiteOld,name,fips,pui,pum

In [4]:
# 1 Find the top 10 states with the highest no.of positive cases

In [5]:
cases = spark.sparkContext.textFile("/public/trendytech/covid19/cases/covid_dataset_cases.csv").map(lambda x: (x.split(",")[1],int(x.split(",")[2])) )

In [6]:
cases.take(3)

[('AP', 2), ('AP', 2), ('HP', 2)]

In [7]:
positive_state = cases.reduceByKey(lambda x,y : x+y).sortBy(lambda x: x[1],False)

In [8]:
positive_state.take(10)

[('WA', 1701),
 ('GA', 1017),
 ('MH', 730),
 ('MI', 61),
 ('CA', 53),
 ('GJ', 35),
 ('BR', 23),
 ('JH', 13),
 ('CG', 8),
 ('RI', 6)]

In [9]:
# 2.Find the total count of people in ICU currently.

In [10]:
icu_curr = spark.sparkContext.textFile("/public/trendytech/covid19/cases/covid_dataset_cases.csv").map(lambda x: int(x.split(",")[7])).sum()

In [11]:
print(icu_curr)

1344


In [12]:
# 3. Find the top 15 States having maximum no.of recovery

In [13]:
state_recover = spark.sparkContext.textFile("/public/trendytech/covid19/cases/covid_dataset_cases.csv").map(lambda x: (x.split(",")[1],int(x.split(",")[11])) )

In [14]:
state_recover.take(5)

[('AP', 34), ('AP', 50), ('HP', 11), ('HP', 8), ('AS', 30)]

In [15]:
state_recover_sum = state_recover.reduceByKey(lambda x,y : x+y).sortBy(lambda x: x[1],False)

In [16]:
state_recover_sum.take(15)

[('WA', 451),
 ('MH', 165),
 ('MI', 101),
 ('GA', 87),
 ('AP', 84),
 ('RI', 72),
 ('BR', 68),
 ('JH', 50),
 ('KA', 43),
 ('AZ', 38),
 ('AS', 30),
 ('GJ', 27),
 ('CA', 23),
 ('HR', 20),
 ('HP', 19)]

In [17]:
# 4.Find the top 3 States having least no.of deaths

In [18]:
state_death = spark.sparkContext.textFile("/public/trendytech/covid19/cases/covid_dataset_cases.csv").map(lambda x: (x.split(",")[1],int(x.split(",")[23])) )

In [19]:
state_death.take(3)

[('AP', 42), ('AP', 45), ('HP', 30)]

In [20]:
death_state_wise = state_death.reduceByKey(lambda x,y : x+y).sortBy(lambda x: x[1])

In [21]:
death_state_wise.take(3)

[('AS', 9), ('JH', 10), ('CG', 31)]

In [22]:
# 5.Find the total number of people hospitalized currently.

In [23]:
hos_curr = spark.sparkContext.textFile("/public/trendytech/covid19/cases/covid_dataset_cases.csv").map(lambda x: int(x.split(",")[5])).sum()

In [24]:
print(hos_curr)

1319


In [25]:
# 6. List the twitter handle and fips code for the top 15 states with the highestnumber of total cases.

In [26]:
states = spark.sparkContext.textFile("/public/trendytech/covid19/states/covid_dataset_states.csv").map(lambda x: (x.split(",")[0],(x.split(",")[5],int(x.split(",")[8]))))

In [27]:
states.take(5)

[('HP', ('@HPCovid', 53)),
 ('AS', ('@ASCovid', 6)),
 ('HR', ('@HRCovid', 9)),
 ('KA', ('@KACovid', 53)),
 ('WA', ('@WACovid', 44))]

In [28]:
total_case = spark.sparkContext.textFile("/public/trendytech/covid19/cases/covid_dataset_cases.csv").map(lambda x: (x.split(",")[1],int(x.split(",")[28])) )

In [29]:
total_case.take(5)

[('AP', 2), ('AP', 2), ('HP', 2), ('HP', 2), ('AS', 2)]

In [30]:
top_total_case = total_case.reduceByKey(lambda x,y:x+y)

In [31]:
top_total_case.take(5)

[('AP', 4), ('HP', 4), ('AS', 2), ('CG', 8), ('BR', 23)]

In [32]:
twit_state = states.join(top_total_case).sortBy(lambda x:x[1][1],False)

In [33]:
twit_state.take(5)

[('WA', (('@WACovid', 44), 2100)),
 ('GA', (('@GACovid', 44), 1034)),
 ('MH', (('@MHCovid', 26), 730)),
 ('CA', (('@CACovid', 4), 515)),
 ('MI', (('@MICovid', 53), 61))]

### Question 3 :

In [35]:
# file /public/trendytech/reviews/trendytech-student-reviews.csv - hdfs

In [36]:
## /data/trendytech/boringwords.txt - in edge node

In [65]:
# !hadoop fs -put /data/trendytech/boringwords.txt /user/itv024128/data

In [2]:
bore_words_rdd = spark.sparkContext.textFile("/user/itv024128/data/boringwords.txt")

In [3]:
bore_words_rdd.take(5)

['shouldnt', 'worrying', 'simplify', 'tidy', 'shouldnt']

In [24]:
bore_words_broadcast = spark.sparkContext.broadcast(set(bore_words_rdd.collect()))  ## convert to set for faster lookup when filtering out

In [5]:
review = spark.sparkContext.textFile("/public/trendytech/reviews/trendytech-student-reviews.csv")

In [23]:
list(bore_words_broadcast.value)[2]

'disturbed'

In [10]:
review.take(2)

['I got to know about this course by recommendation from one of my senior and i would say that the course has Excellent lectures(content) that explored my mind.',
 'The Sumit is very much passionate about teaching Big Data and his style of teaching is very unique and engaging the flow of course content.']

In [11]:
flatmap = review.flatMap(lambda x: (x.split(" "))).map(lambda x : x.lower())

In [12]:
flatmap.take(5)

['i', 'got', 'to', 'know', 'about']

In [13]:
 no_boring =  flatmap.filter(lambda x: x not in bore_words_broadcast.value)  ## anti join using filter

In [14]:
no_boring.take(5)

['lectures(content)', 'explored', 'mind.', 'sumit', 'passionate']

In [15]:
words1 = no_boring.map(lambda x: (x,1))

In [16]:
words1.take(5)

[('lectures(content)', 1),
 ('explored', 1),
 ('mind.', 1),
 ('sumit', 1),
 ('passionate', 1)]

In [17]:
wc1 = words1.reduceByKey(lambda x,y : x+y).sortBy(lambda x: x[1],False)

In [18]:
wc1.take(5)

[('data', 201), ('sumit', 109), ('trendytech', 67), ('', 64), ('data.', 34)]